In [2]:
import os
import shutil
from pathlib import Path

In [3]:
SOURCE_DIR = Path('data')      # original downloaded dataset
DEST_DIR   = Path('binary_data')   # new binary folder structure


In [4]:
LABEL_MAP = {
    'angry':    'irritated',
    'disgust':  'irritated',
    'fear':     'irritated',
    'happy':    'neutral',
    'neutral':  'neutral',
    'sad':      'irritated',
    'surprise': 'irritated',
}

In [5]:
def remap_split(split):
    """Copy images from fer2013/{split}/{emotion}/ into binary_data/{split}/{neutral|irritated}/"""
    counts = {'neutral': 0, 'irritated': 0}

    for emotion_folder in (SOURCE_DIR / split).iterdir():
        if not emotion_folder.is_dir():
            continue

        emotion_name = emotion_folder.name.lower()

        if emotion_name not in LABEL_MAP:
            print(f"  Skipping unknown folder: {emotion_name}")
            continue

        binary_label = LABEL_MAP[emotion_name]
        dest_folder  = DEST_DIR / split / binary_label
        dest_folder.mkdir(parents=True, exist_ok=True)

        images = list(emotion_folder.glob('*.jpg')) + \
                 list(emotion_folder.glob('*.png')) + \
                 list(emotion_folder.glob('*.jpeg'))

        for img_path in images:
            # Prefix filename with source emotion to avoid name collisions
            dest_name = f"{emotion_name}_{img_path.name}"
            shutil.copy(img_path, dest_folder / dest_name)
            counts[binary_label] += 1

    print(f"  {split}: neutral={counts['neutral']}, irritated={counts['irritated']}")
    return counts

In [6]:
if __name__ == '__main__':
    print("Remapping dataset to binary classes...\n")

    train_counts = remap_split('train')
    test_counts  = remap_split('test')

    total_neutral   = train_counts['neutral']   + test_counts['neutral']
    total_irritated = train_counts['irritated'] + test_counts['irritated']

    print(f"""
✅ Done! Binary dataset created in binary_data/

  NEUTRAL   : {total_neutral:,} images (happy + neutral)
  IRRITATED : {total_irritated:,} images (angry + disgust + fear + sad + surprise)

  binary_data/
    train/neutral/     {train_counts['neutral']:,} images
    train/irritated/   {train_counts['irritated']:,} images
    test/neutral/      {test_counts['neutral']:,} images
    test/irritated/    {test_counts['irritated']:,} images
""")

Remapping dataset to binary classes...

  train: neutral=12180, irritated=16529
  test: neutral=3007, irritated=4171

✅ Done! Binary dataset created in binary_data/

  NEUTRAL   : 15,187 images (happy + neutral)
  IRRITATED : 20,700 images (angry + disgust + fear + sad + surprise)

  binary_data/
    train/neutral/     12,180 images
    train/irritated/   16,529 images
    test/neutral/      3,007 images
    test/irritated/    4,171 images

